# Workshop 1 — Transfer Learning + Confusion Matrix

**โจทย์:** ทำ transfer learning กับชุดข้อมูลภาพ แล้วแสดงผลลัพธ์ด้วย confusion matrix

**สิ่งที่จะทำในโน้ตบุ๊กนี้**

| ขั้น | ทำอะไร |
|---|---|
| 1 | โหลดชุดข้อมูลดอกไม้ 5 ชนิด (3,670 ภาพ) |
| 2 | เตรียม data pipeline: แบ่ง train/validation, cache, prefetch |
| 3 | สร้าง baseline — CNN ที่ฝึกเองจากศูนย์ ไว้เป็นตัวเทียบ |
| 4 | Transfer learning ขั้นที่ 1 — feature extraction ด้วย MobileNetV2 |
| 5 | Transfer learning ขั้นที่ 2 — fine-tuning |
| 6 | **Confusion matrix** + classification report + ดูภาพที่ทายผิด |

**โมเดลที่เลือก:** `MobileNetV2` ที่ผ่านการฝึกมาแล้วบน ImageNet (1.4 ล้านภาพ 1,000 คลาส) เลือกตัวนี้เพราะเบา (3.5M พารามิเตอร์) รันบน CPU ของ MacBook ได้สบาย ประมาณ 17 วินาทีต่อ epoch

---

## ขั้นที่ 0 — โหลดไลบรารีและตรวจสภาพแวดล้อม

รันเซลล์นี้ก่อนเสมอ ถ้า `GPU ที่เจอ` เป็นลิสต์ว่าง แปลว่ารันด้วย CPU ซึ่งชุดข้อมูลขนาดนี้ยังไหวสบาย

In [ ]:
import pathlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version :", tf.__version__)
print("GPU ที่เจอ          :", tf.config.list_physical_devices('GPU'))

# ตรึง seed เพื่อให้ผลลัพธ์ทำซ้ำได้
keras.utils.set_random_seed(42)

---

## ขั้นที่ 1 — โหลดชุดข้อมูล

ใช้ชุด **flower_photos** ดอกไม้ 5 ชนิด รวม 3,670 ภาพ (ชุดเดียวกับที่มีบน Kaggle ในชื่อ *Flowers Recognition* แต่โหลดจาก Google Storage ได้โดยตรง ไม่ต้องใช้ API token)

`untar=True` จะแตกไฟล์ให้อัตโนมัติ และถ้าเคยโหลดแล้วจะข้ามไปใช้ของเดิมในเครื่อง

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
path = keras.utils.get_file("flower_photos", origin=url, untar=True)

# ไฟล์ที่แตกออกมามีโฟลเดอร์ flower_photos ซ้อนอยู่ข้างในอีกชั้น
data_dir = pathlib.Path(path)
if not (data_dir / 'daisy').exists():
    data_dir = data_dir / 'flower_photos'

print("โฟลเดอร์ข้อมูล:", data_dir)
print()
total = 0
for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    n = len(list(folder.glob('*.jpg')))
    total += n
    print(f"  {folder.name:<12} {n:>5} ภาพ")
print(f"  {'รวม':<12} {total:>5} ภาพ")

**โครงสร้างโฟลเดอร์สำคัญมาก** — Keras อ่านชื่อโฟลเดอร์เป็นชื่อคลาสโดยอัตโนมัติ

```
flower_photos/
├── daisy/        (633 ภาพ)
├── dandelion/    (898 ภาพ)
├── roses/        (641 ภาพ)
├── sunflowers/   (699 ภาพ)
└── tulips/       (799 ภาพ)
```

สังเกตว่าจำนวนภาพแต่ละคลาสไม่เท่ากัน (633 ถึง 898) เรียกว่า *class imbalance* เล็กน้อย ยังไม่ถึงขั้นต้องแก้ แต่จำไว้ตอนอ่าน confusion matrix

---

## ขั้นที่ 2 — สร้าง data pipeline

### ขั้นที่ 2.1 — แบ่ง train / validation

`image_dataset_from_directory` ทำสามอย่างพร้อมกัน: อ่านไฟล์ภาพ, ย่อขนาดให้เท่ากัน, และติด label จากชื่อโฟลเดอร์

- `image_size=(224, 224)` — MobileNetV2 ถูกฝึกมาที่ขนาดนี้ ใช้ให้ตรงกันจะได้ผลดีที่สุด
- `validation_split=0.2` — กันไว้ 20% (734 ภาพ) สำหรับวัดผล ไม่ให้โมเดลเห็นตอนฝึก
- `seed=42` **ต้องเป็นเลขเดียวกันทั้งสองชุด** ไม่งั้นภาพจะปนกันระหว่าง train กับ validation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
print()
print("คลาสทั้งหมด:", class_names)

### ขั้นที่ 2.2 — ดูภาพจริงก่อนเสมอ

ก่อนฝึกโมเดลควรดูภาพสักชุดเสมอ เพื่อยืนยันว่าภาพกับ label ตรงกันจริง ถ้าข้อมูลผิดตั้งแต่ต้น ฝึกไปเท่าไรก็ไม่มีทางถูก

In [ ]:
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    plt.title(class_names[labels[i]], fontsize=10)
    plt.axis('off')
plt.suptitle("ตัวอย่างภาพ 10 ภาพแรกจาก batch แรก", fontsize=13)
plt.tight_layout()
plt.show()

print("รูปทรงของ batch :", images.shape)   # (32, 224, 224, 3)
print("ช่วงค่าพิกเซล   :", float(images.numpy().min()), "ถึง", float(images.numpy().max()))

**สังเกตช่วงค่าพิกเซล** ตอนนี้ยังเป็น 0-255 อยู่ เรายัง**ไม่**หารด้วย 255 ที่ตรงนี้ เพราะ MobileNetV2 มีฟังก์ชัน `preprocess_input` ของตัวเองที่แปลงเป็นช่วง -1 ถึง 1 ซึ่งเราจะใส่ไว้เป็นชั้นแรกของโมเดลแทน

### ขั้นที่ 2.3 — cache และ prefetch

สองบรรทัดนี้ทำให้ฝึกเร็วขึ้นมาก

- `cache()` — อ่านภาพจากดิสก์แค่รอบเดียว epoch ถัดไปหยิบจาก RAM
- `prefetch()` — ขณะที่ GPU/CPU กำลังคำนวณ batch ปัจจุบัน ให้เตรียม batch ถัดไปไว้พร้อมกัน ไม่ต้องรอกัน
- `shuffle()` — สลับลำดับภาพในแต่ละ epoch กันไม่ให้โมเดลจำลำดับ

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

print("เตรียม pipeline เรียบร้อย")

### ขั้นที่ 2.4 — Data augmentation

ข้อมูลเรามีแค่ 2,936 ภาพสำหรับฝึก ซึ่งน้อยมากสำหรับงานภาพ วิธีแก้คือ **augmentation** — พลิก/หมุน/ซูมภาพแบบสุ่มทุกครั้งที่โมเดลเห็น ทำให้โมเดลเจอภาพที่ไม่ซ้ำเดิมเลย ช่วยลด overfitting

ชั้นพวกนี้จะทำงาน**เฉพาะตอนฝึก** ตอน evaluate หรือ predict Keras จะปิดให้เองอัตโนมัติ

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='data_augmentation')

# ดูผลของ augmentation กับภาพเดียวกัน 8 ครั้ง
sample = images[0]
plt.figure(figsize=(12, 6))
for i in range(8):
    aug = data_augmentation(tf.expand_dims(sample, 0), training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.cast(aug[0], tf.uint8))
    plt.axis('off')
plt.suptitle(f"ภาพเดียวกัน ('{class_names[labels[0]]}') ผ่าน augmentation 8 ครั้ง", fontsize=13)
plt.tight_layout()
plt.show()

---

## ขั้นที่ 3 — Baseline: CNN ที่ฝึกเองจากศูนย์

ก่อนจะไป transfer learning เราต้องมีตัวเทียบก่อน ไม่งั้นจะไม่รู้ว่า transfer learning ดีขึ้นจริงแค่ไหน

โมเดลนี้เริ่มจากน้ำหนักสุ่มทั้งหมด ต้องเรียนรู้ตั้งแต่ "เส้นขอบคืออะไร" ด้วยภาพแค่ 2,936 ภาพ

In [ ]:
baseline = keras.Sequential([
    layers.Input((224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax'),
], name='baseline_cnn')

baseline.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

baseline.summary()

In [ ]:
history_baseline = baseline.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

### ขั้นที่ 3.1 — ฟังก์ชันวาดกราฟ

เขียนไว้ครั้งเดียว ใช้ซ้ำกับทุกโมเดล

In [ ]:
def plot_history(history, title="", initial_epoch=0):
    h = history.history
    epochs = range(initial_epoch, initial_epoch + len(h['loss']))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))

    ax[0].plot(epochs, h['loss'], label='train', linewidth=2)
    ax[0].plot(epochs, h['val_loss'], label='validation', linewidth=2)
    ax[0].set_title(f'{title} — loss')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss')
    ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(epochs, h['accuracy'], label='train', linewidth=2)
    ax[1].plot(epochs, h['val_accuracy'], label='validation', linewidth=2)
    ax[1].set_title(f'{title} — accuracy')
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy')
    ax[1].legend(); ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_history(history_baseline, "Baseline CNN")

baseline_acc = max(history_baseline.history['val_accuracy'])
print(f"Baseline CNN — validation accuracy สูงสุด: {baseline_acc:.4f}")

**อ่านผล:** baseline จะได้ราว 60-70% ซึ่งดีกว่าเดาสุ่ม (20%) แต่ไม่น่าประทับใจ กราฟ loss ของ validation มักจะเริ่มแบนหรือแกว่ง เพราะข้อมูลน้อยเกินกว่าจะเรียนรู้ฟีเจอร์ภาพที่ดีได้เอง

**นี่คือเหตุผลที่ต้องใช้ transfer learning**

---

## ขั้นที่ 4 — Transfer Learning ขั้นที่ 1: Feature Extraction

### แนวคิด

MobileNetV2 ถูกฝึกมาบน ImageNet — 1.4 ล้านภาพ 1,000 คลาส ระหว่างนั้นมันได้เรียนรู้การมองภาพไปแล้ว ชั้นต้น ๆ จับเส้นขอบและสี ชั้นกลางจับลวดลายและพื้นผิว ชั้นลึกจับรูปทรงที่ซับซ้อน **ความรู้พวกนี้ใช้กับภาพดอกไม้ได้เลย** ถึงแม้ ImageNet จะไม่ได้ฝึกมาเพื่อแยกดอกไม้ 5 ชนิดนี้โดยตรง

เราจึงตัดหัวเดิม (ชั้นที่ทำนาย 1,000 คลาส) ทิ้ง เก็บส่วนที่เหลือไว้เป็น *feature extractor* แล้วต่อหัวใหม่ที่ทำนาย 5 คลาสของเรา

### ส่วนประกอบของโมเดล

| ชั้น | หน้าที่ |
|---|---|
| `data_augmentation` | สุ่มพลิก/หมุน/ซูม (ทำงานเฉพาะตอนฝึก) |
| `preprocess_input` | แปลงพิกเซล 0-255 → -1 ถึง 1 ตามที่ MobileNetV2 คาดหวัง |
| `base_model` | MobileNetV2 **ตรึงน้ำหนักไว้** (`trainable = False`) |
| `GlobalAveragePooling2D` | ยุบ feature map 7×7×1280 ให้เหลือเวกเตอร์ 1280 ตัว |
| `Dropout(0.2)` | สุ่มปิด 20% ของ neuron กัน overfit |
| `Dense(5, softmax)` | หัวใหม่ของเรา ทำนาย 5 คลาส |

**`base_model.trainable = False` คือหัวใจของขั้นนี้** เราฝึกเฉพาะชั้น Dense ท้ายสุด น้ำหนัก 2.2 ล้านตัวของ MobileNetV2 ไม่ถูกแตะเลย

In [ ]:
base_model = keras.applications.MobileNetV2(
    include_top=False,           # ตัดหัวเดิมที่ทำนาย 1,000 คลาสทิ้ง
    weights='imagenet',          # โหลดน้ำหนักที่ฝึกมาแล้ว
    input_shape=(224, 224, 3),
)
base_model.trainable = False     # ตรึงน้ำหนักเดิมไว้ทั้งหมด

print("จำนวนชั้นใน MobileNetV2 :", len(base_model.layers))
print("พารามิเตอร์ทั้งหมด      :", f"{base_model.count_params():,}")

In [ ]:
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)          # training=False ตรึง BatchNorm ไว้ด้วย
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = keras.Model(inputs, outputs, name='mobilenetv2_transfer')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

**อ่าน summary:** ดูบรรทัด `Trainable params` จะเห็นแค่ประมาณ 6,400 ตัว (1280 × 5 + 5) ส่วน `Non-trainable params` มี 2.2 ล้าน — นั่นคือ MobileNetV2 ที่เราตรึงไว้

เราฝึกพารามิเตอร์แค่ 0.3% ของโมเดลทั้งหมด นี่คือเหตุผลที่ transfer learning เร็วและใช้ข้อมูลน้อย

In [ ]:
EPOCHS_FE = 10

history_fe = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FE,
)

In [ ]:
plot_history(history_fe, "Feature Extraction")

fe_acc = max(history_fe.history['val_accuracy'])
print(f"Feature extraction — validation accuracy สูงสุด: {fe_acc:.4f}")
print(f"เทียบกับ baseline CNN                        : {baseline_acc:.4f}")
print(f"ดีขึ้น                                        : {(fe_acc - baseline_acc) * 100:+.2f} จุด")

**สังเกต:** accuracy พุ่งไปเกิน 85% ตั้งแต่ epoch แรก ๆ ทั้งที่ฝึกพารามิเตอร์แค่ 6,400 ตัว เทียบกับ baseline ที่ฝึก 100,000+ ตัวแล้วยังได้แค่ 60-70%

ความรู้จาก ImageNet ทำงานได้จริง

---

## ขั้นที่ 5 — Transfer Learning ขั้นที่ 2: Fine-tuning

### แนวคิด

ตอนนี้หัวใหม่ของเราฝึกมาดีแล้ว ขั้นต่อไปคือ "ปลดล็อก" ชั้นท้าย ๆ ของ MobileNetV2 ให้ปรับตัวเข้ากับดอกไม้โดยเฉพาะ

**กฎสำคัญสองข้อ**

1. **ปลดล็อกเฉพาะชั้นท้าย** — ชั้นต้น ๆ จับเส้นขอบและสีซึ่งใช้ได้กับภาพทุกชนิดอยู่แล้ว ไม่ควรแตะ ส่วนชั้นท้ายจับรูปทรงเฉพาะทางซึ่งควรปรับ
2. **ลด learning rate ลงมาก ๆ** — จาก `1e-3` เหลือ `1e-5` (น้อยลง 100 เท่า) ถ้าใช้ค่าเดิม gradient ก้อนใหญ่จะทำลายน้ำหนักดี ๆ ที่ ImageNet ฝึกมาจนพัง เรียกว่า *catastrophic forgetting*

**ต้อง `compile()` ใหม่ทุกครั้งหลังเปลี่ยน `trainable`** ไม่งั้น Keras จะยังใช้ค่าเดิมที่ compile ไว้ก่อนหน้า

In [ ]:
base_model.trainable = True

# ปลดล็อกเฉพาะ 30 ชั้นท้าย ที่เหลือตรึงไว้เหมือนเดิม
FINE_TUNE_FROM = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

# BatchNorm ต้องตรึงไว้เสมอตอน fine-tune ไม่งั้นสถิติ mean/variance จะเพี้ยน
for layer in base_model.layers[FINE_TUNE_FROM:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# compile ใหม่ด้วย learning rate ที่ต่ำมาก — ขั้นตอนนี้ห้ามข้าม
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = sum(int(tf.size(w)) for w in model.trainable_weights)
frozen = sum(int(tf.size(w)) for w in model.non_trainable_weights)
print(f"ปลดล็อกตั้งแต่ชั้นที่ {FINE_TUNE_FROM} เป็นต้นไป")
print(f"พารามิเตอร์ที่ฝึกได้ : {trainable:,}")
print(f"พารามิเตอร์ที่ตรึงไว้ : {frozen:,}")

In [ ]:
EPOCHS_FT = 10

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FE + EPOCHS_FT,
    initial_epoch=EPOCHS_FE,      # นับ epoch ต่อจากรอบที่แล้ว กราฟจะได้ต่อกัน
)

### ขั้นที่ 5.1 — กราฟรวมสองเฟส

วาดต่อกันจะเห็นชัดว่าจุดที่เริ่ม fine-tune accuracy กระโดดขึ้นอีกขั้น

In [ ]:
h_all = {
    k: history_fe.history[k] + history_ft.history[k]
    for k in history_fe.history
}

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(h_all['loss'], label='train', linewidth=2)
ax[0].plot(h_all['val_loss'], label='validation', linewidth=2)
ax[0].axvline(EPOCHS_FE - 1, color='red', linestyle='--', label='เริ่ม fine-tune')
ax[0].set_title('loss ตลอดสองเฟส'); ax[0].set_xlabel('epoch')
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(h_all['accuracy'], label='train', linewidth=2)
ax[1].plot(h_all['val_accuracy'], label='validation', linewidth=2)
ax[1].axvline(EPOCHS_FE - 1, color='red', linestyle='--', label='เริ่ม fine-tune')
ax[1].set_title('accuracy ตลอดสองเฟส'); ax[1].set_xlabel('epoch')
ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

ft_acc = max(history_ft.history['val_accuracy'])
print(f"Feature extraction : {fe_acc:.4f}")
print(f"Fine-tuning        : {ft_acc:.4f}  ({(ft_acc - fe_acc) * 100:+.2f} จุด)")

---

## ขั้นที่ 6 — Confusion Matrix

### ทำไมต้องดู confusion matrix

Accuracy ตัวเดียวบอกแค่ "ถูกกี่เปอร์เซ็นต์" แต่ไม่บอกว่า**ผิดตรงไหน** ถ้าโมเดลได้ 90% แต่พลาดกับคลาสสำคัญคลาสเดียวหมดเลย ตัวเลข 90% นั้นก็หลอกเรา

Confusion matrix บอกครบ: แถว = คำตอบจริง, คอลัมน์ = สิ่งที่โมเดลทาย ช่องแนวทแยงคือทายถูก ช่องนอกแนวทแยงคือทายผิด และบอกด้วยว่าสับสนกับคลาสไหน

### ขั้นที่ 6.1 — เก็บคำทำนายทั้ง validation set

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

y_true = []
y_pred = []
y_prob = []

for batch_images, batch_labels in val_ds:
    probs = model.predict(batch_images, verbose=0)
    y_true.extend(batch_labels.numpy())
    y_pred.extend(probs.argmax(axis=1))
    y_prob.extend(probs.max(axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print(f"ทำนายทั้งหมด {len(y_true)} ภาพ")
print(f"ถูก {int((y_true == y_pred).sum())} ภาพ  ผิด {int((y_true != y_pred).sum())} ภาพ")
print(f"accuracy = {(y_true == y_pred).mean():.4f}")

### ขั้นที่ 6.2 — Confusion matrix แบบนับจำนวน

ตัวเลขในช่องคือจำนวนภาพ

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', colorbar=True, values_format='d')
ax.set_title('Confusion Matrix — จำนวนภาพ', fontsize=14, pad=15)
ax.set_xlabel('โมเดลทายว่า (Predicted)', fontsize=11)
ax.set_ylabel('คำตอบจริง (True)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### ขั้นที่ 6.3 — Confusion matrix แบบเปอร์เซ็นต์ตามแถว

เพราะแต่ละคลาสมีจำนวนภาพไม่เท่ากัน การดูเป็นเปอร์เซ็นต์ต่อแถวจะเทียบกันได้ตรงกว่า

**อ่านยังไง:** ค่าในช่อง (แถว A, คอลัมน์ B) = "ภาพที่จริง ๆ เป็น A มีกี่ % ที่โมเดลทายว่าเป็น B" แต่ละแถวรวมกันได้ 100%

In [ ]:
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)

ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel('โมเดลทายว่า (Predicted)', fontsize=11)
ax.set_ylabel('คำตอบจริง (True)', fontsize=11)
ax.set_title('Confusion Matrix — เปอร์เซ็นต์ตามแถว', fontsize=14, pad=15)

# เขียนตัวเลขลงในทุกช่อง สีขาวบนพื้นเข้ม สีดำบนพื้นอ่อน
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, f"{cm_norm[i, j]:.1%}\n({cm[i, j]})",
                ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### ขั้นที่ 6.4 — Classification report

ตารางนี้แยกให้ดูเป็นรายคลาส

- **precision** — ในบรรดาภาพที่โมเดลทายว่าเป็นคลาสนี้ ถูกจริงกี่ % *(ทายมั่วแค่ไหน)*
- **recall** — ในบรรดาภาพที่เป็นคลาสนี้จริง โมเดลจับได้กี่ % *(พลาดไปเยอะไหม)*
- **f1-score** — ค่าเฉลี่ยแบบ harmonic ของสองตัวบน ใช้ดูภาพรวมของคลาสนั้น
- **support** — จำนวนภาพจริงของคลาสนั้นใน validation set

In [ ]:
print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=3,
))

### ขั้นที่ 6.5 — คู่ที่โมเดลสับสนมากที่สุด

ดึงช่องนอกแนวทแยงที่มีค่าสูงสุดออกมาเรียงลำดับ จะได้รู้ว่าควรไปแก้ตรงไหนก่อน

In [ ]:
pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], cm_norm[i, j], class_names[i], class_names[j]))

pairs.sort(reverse=True)

print(f"{'จริง':<14}{'ทายเป็น':<14}{'จำนวน':>7}{'% ของคลาสนั้น':>16}")
print("=" * 52)
for count, ratio, true_c, pred_c in pairs[:8]:
    print(f"{true_c:<14}{pred_c:<14}{count:>7}{ratio:>15.1%}")

### ขั้นที่ 6.6 — ดูภาพที่ทายผิดจริง ๆ

ตัวเลขบอกว่าผิดตรงไหน แต่ภาพบอกว่า**ทำไม** หลายครั้งจะพบว่าภาพที่โมเดลทายผิดนั้นคนก็ตอบยากเหมือนกัน หรือบางภาพ label เดิมก็ผิดเอง

In [ ]:
# เก็บภาพจาก validation set ทั้งหมดเรียงตามลำดับเดิม (val_ds ไม่ถูก shuffle)
val_images = np.concatenate([imgs.numpy() for imgs, _ in val_ds], axis=0)

wrong_idx = np.where(y_true != y_pred)[0]
# เรียงจากภาพที่โมเดลมั่นใจมากที่สุดแต่ตอบผิด — พวกนี้คือความผิดพลาดที่น่าสนใจที่สุด
wrong_idx = wrong_idx[np.argsort(-y_prob[wrong_idx])]

n_show = min(12, len(wrong_idx))
plt.figure(figsize=(14, 9))
for k in range(n_show):
    idx = wrong_idx[k]
    plt.subplot(3, 4, k + 1)
    plt.imshow(val_images[idx].astype('uint8'))
    plt.title(
        f"จริง: {class_names[y_true[idx]]}\n"
        f"ทาย: {class_names[y_pred[idx]]} ({y_prob[idx]:.0%})",
        fontsize=9, color='crimson',
    )
    plt.axis('off')

plt.suptitle(f"ภาพที่ทายผิด {n_show} ภาพ (เรียงตามความมั่นใจของโมเดล)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"ทายผิดทั้งหมด {len(wrong_idx)} ภาพ จาก {len(y_true)} ภาพ")

---

## ขั้นที่ 7 — สรุปผล

In [ ]:
print(f"{'วิธี':<34}{'val accuracy':<16}{'trainable params'}")
print("=" * 70)

baseline_trainable = sum(int(tf.size(w)) for w in baseline.trainable_weights)
print(f"{'CNN ฝึกเองจากศูนย์ (baseline)':<30}{baseline_acc:<16.4f}{baseline_trainable:,}")
print(f"{'Transfer — feature extraction':<32}{fe_acc:<16.4f}{6405:,}")
print(f"{'Transfer — fine-tuning':<34}{ft_acc:<16.4f}{trainable:,}")
print("=" * 70)
print()
print(f"transfer learning ดีกว่า baseline : {(ft_acc - baseline_acc) * 100:+.2f} จุด")
print(f"fine-tuning ดีกว่า feature extraction : {(ft_acc - fe_acc) * 100:+.2f} จุด")

### สิ่งที่ได้จากงานนี้

**1. Transfer learning ชนะการฝึกเองขาดลอย**

Baseline CNN ฝึกพารามิเตอร์แสนกว่าตัวจากศูนย์ ได้ราว 60-70% ส่วน feature extraction ฝึกแค่ 6,405 ตัว ได้เกิน 85% ตั้งแต่ epoch แรก ๆ เพราะความรู้เรื่อง "การมองภาพ" ที่ MobileNetV2 เรียนมาจาก ImageNet 1.4 ล้านภาพนั้นถ่ายทอดมาใช้ได้เลย

**2. Fine-tuning ต่อยอดได้อีก แต่ต้องระวัง**

การปลดล็อกชั้นท้ายให้ปรับตัวเข้ากับดอกไม้โดยเฉพาะช่วยเพิ่ม accuracy ขึ้นได้อีก แต่ต้องลด learning rate ลง 100 เท่า ถ้าใช้ `1e-3` เหมือนเดิม น้ำหนักที่ดีอยู่แล้วจะถูกทำลาย

**3. Confusion matrix เล่าสิ่งที่ accuracy ไม่เล่า**

คู่ที่สับสนกันมากที่สุดมักเป็น **roses กับ tulips** เพราะดอกไม้สองชนิดนี้มีสีแดง/ชมพูเหมือนกัน กลีบซ้อนกันเป็นถ้วยคล้ายกัน และในภาพระยะไกลแทบแยกไม่ออกแม้แต่คน ส่วน **sunflowers** มักได้คะแนนสูงสุดเพราะสีเหลืองกับรูปทรงจานกลางเด่นชัด ไม่เหมือนใคร

ข้อมูลนี้บอกทิศทางการปรับปรุงต่อได้ตรงจุด — ถ้าจะเพิ่ม accuracy ควรหาภาพ roses/tulips เพิ่ม หรือใช้ภาพความละเอียดสูงขึ้น มากกว่าจะไปเพิ่มภาพ sunflowers ที่โมเดลทำได้ดีอยู่แล้ว

### ลองต่อเองได้

- เปลี่ยน base model เป็น `EfficientNetV2B0` หรือ `ResNet50V2` แล้วเทียบผล (อย่าลืมเปลี่ยน `preprocess_input` ให้ตรงกับโมเดลนั้นด้วย)
- ปรับ `FINE_TUNE_FROM` ให้ปลดล็อกมาก/น้อยกว่า 30 ชั้น
- เพิ่ม `class_weight` ตอน `fit()` เพื่อชดเชยที่แต่ละคลาสมีภาพไม่เท่ากัน
- ใช้ `EarlyStopping` และ `ReduceLROnPlateau` callback แทนการกำหนด epoch ตายตัว